# “How does a deep learning framework compute gradients automatically for any arbitrary computation graph?”

### Autograd is an automatic differentiation engine that computes gradients by applying the chain rule over a dynamically constructed computational grap

Key ideas:

    Graph is built at runtime
    Gradients flow backward
    Uses reverse-mode differentiation

In [ ]:
# Manual Gradient

Example function
    𝑦 = 𝑥^2 + 3 𝑥
 
Manual derivative
    𝑑𝑦/𝑑𝑥 = 2𝑥 + 3

“ It is easy for one variable. Impossible to scale to millions of parameters.”

In [ ]:
# What Autograd Tracks

| Component | Meaning                |
| --------- | ---------------------- |
| Tensor    | Holds data             |
| Operation | Produces output tensor |
| Graph     | Connects operations    |

Autograd does NOT differentiate code. It differentiates operations
   

In [19]:
# Autograd

import torch

x = torch.tensor(2.0, requires_grad=True)

y = x**2 + 3*x
y.backward()
print(y)

print(x.grad)

# (it had just did this ---> 2(2)+3=7)

tensor(10., grad_fn=<AddBackward0>)
tensor(7.)


In [ ]:
# what happens internally ?

1 x is a leaf tensor
2 Operations create a dynamic graph
3 y stores a reference to its creator (grad_fn)
4 backward():

    Starts from scalar output
    Applies chain rule
    Accumulates gradients in .grad

This is reverse-mode autodiff !

In [2]:
print(y.grad_fn)
print(y.grad_fn.next_functions)

        y
        |
   AddBackward0
     /      \
PowBackward0  MulBackward0
    (x^^2)      (3X)

((<PowBackward0 object at 0x000001876DFBCB20>, 0), (<MulBackward0 object at 0x000001876DFBC820>, 0))


In [ ]:
# Chain Rule

    z=(x^2+3x)^2

In [3]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x
z = y**2

z.backward()
print(x.grad)

### Autograd is just a very fast, very disciplined chain rule engine

tensor(140.)


In [ ]:
So what we have done is 

dz/dx = dz/dy * dy/dx

In [ ]:
Why Reverse-Mode ?

| Mode         | Used for                  |
| ------------ | ------------------------- |
| Forward-mode | Few inputs, many outputs  |
| Reverse-mode | Many parameters, one loss |

Deep learning = millions of parameters + scalar loss
➡ Reverse-mode is optimal

In [4]:
# Gradient Accumulation
x = torch.tensor(2.0, requires_grad=True)

y1 = x**2
y1.backward()
print(x.grad)  # 4

y2 = x**3
y2.backward()
print(x.grad)  # 4 + 12 = 16

## Gradients accumulate by default

tensor(4.)
tensor(16.)


In [5]:
# Zeroing Gradients (Why optimizers do it)

x.grad.zero_()

## This is why optimizer.zero_grad() exists

tensor(0.)

In [16]:
# Autograd with Vectors

W = torch.randn(3, requires_grad=True)
x = torch.tensor([1.0, 2.0, 3.0])
y = (W * x).sum()

y.backward() # Backpropagation starts here; Compute dou y/ dou W for every element in W
print(W.grad)

# Before and after backward():

#     backward() does NOT change values
#     It only computes gradients

# If y = W·x, then the gradient of y w.r.t W is x.

tensor(-2.0610, grad_fn=<SumBackward0>)
tensor(-2.0610, grad_fn=<SumBackward0>)
tensor([1., 2., 3.])


In [17]:
x = torch.tensor(2.0, requires_grad=True)
print(x)
y = x.detach() ** 2
print(y)

tensor(2., requires_grad=True)
tensor(4.)


In [ ]:
# https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html